In [26]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Continue from here


In [ ]:
import os
import pandas as pd

# Naming convention: data/eval_results/{model}_{ablation}.csv
#   required columns: original_text, reference_summary, generated_summary
RESULTS_DIR = "data/eval_results"
MODELS = ["base", "fine", "distilled"]
ABLATIONS = ["raw", "coref", "textrank", "coref+tr"]

def safe(name):
    return name.replace("+", "_plus_")

results = {}
missing = []

for model in MODELS:
    for ab in ABLATIONS:
        path = f"{RESULTS_DIR}/{model}_{safe(ab)}.csv"
        if os.path.exists(path):
            results[(model, ab)] = pd.read_csv(path)
        else:
            missing.append((model, ab, path))

print(f"Loaded {len(results)} / 12 combinations")
if missing:
    print("\nMissing (will be skipped):")
    for m, a, p in missing:
        print(f"  - {m} x {a:<10} (expected: {p})")

In [ ]:
for (model, ab), df in results.items():
    print(f"{model:<10} x {ab:<10} : {len(df):4} rows")

In [ ]:
# Show one example loaded combo (if any exist)
if results:
    first_key = next(iter(results))
    print(f"Showing: {first_key}")
    display(results[first_key].head())

In [ ]:
# Build per-combo (predictions, references) pairs
combo_data = {}

for (model, ab), df in results.items():
    combo_data[(model, ab)] = (
        df["generated_summary"].fillna("").astype(str).tolist(),
        df["reference_summary"].fillna("").astype(str).tolist(),
    )

# Sanity: same test split implies same reference_summary across combos that share the
# same input row. Quick check using the 'base' model as anchor.
anchor_refs = None
for (model, ab), (_, refs) in combo_data.items():
    if anchor_refs is None:
        anchor_refs = refs
        anchor_key = (model, ab)
        continue
    if len(refs) != len(anchor_refs):
        print(f"WARNING: {model}x{ab} has {len(refs)} rows but {anchor_key} has {len(anchor_refs)}")
    elif refs[:5] != anchor_refs[:5]:
        print(f"WARNING: {model}x{ab} reference order differs from {anchor_key} -- split may not be aligned")

print(f"\nReady: {len(combo_data)} combos prepared for metric computation")

In [31]:
!pip install bert_score

In [ ]:
import pandas as pd
from bert_score import score

print("Calculating BertScore for each (model, ablation) combination...")

bert_rows = []
per_sample_bert = {}

for (model, ab), (y_pred, y_true) in combo_data.items():
    print(f"\n[{model} x {ab}] computing BERTScore over {len(y_pred)} samples...")
    P, R, F1 = score(y_pred, y_true, lang="en", verbose=False)
    bert_rows.append({
        "Model": model,
        "Ablation": ab,
        "Precision": P.mean().item(),
        "Recall": R.mean().item(),
        "F1": F1.mean().item(),
    })
    per_sample_bert[(model, ab)] = {
        "precision": P.tolist(),
        "recall": R.tolist(),
        "f1": F1.tolist(),
    }

system_scores_df = pd.DataFrame(bert_rows)
print("\nAggregated System-Level BERTScores:")
display(system_scores_df)

os.makedirs("data/eval_results", exist_ok=True)
system_scores_df.to_csv("data/eval_results/bert_system_scores.csv", index=False)
print("Saved: data/eval_results/bert_system_scores.csv")

In [33]:
!pip install -q evaluate

In [34]:
!pip install rouge_score

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

rouge_rows = []

for (model, ab), (y_pred, y_true) in combo_data.items():
    print(f"[{model} x {ab}] computing ROUGE...")
    r = rouge.compute(predictions=y_pred, references=y_true)
    rouge_rows.append({
        "Model": model,
        "Ablation": ab,
        "ROUGE-1":    float(r["rouge1"]),
        "ROUGE-2":    float(r["rouge2"]),
        "ROUGE-L":    float(r["rougeL"]),
        "ROUGE-Lsum": float(r["rougeLsum"]),
    })

system_rouge_df = pd.DataFrame(rouge_rows)
print("\nAggregated System-Level ROUGE Scores:")
display(system_rouge_df)

system_rouge_df.to_csv("data/eval_results/rouge_system_scores.csv", index=False)
print("Saved: data/eval_results/rouge_system_scores.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

bert_melted = system_scores_df.melt(
    id_vars=["Model", "Ablation"],
    value_vars=["Precision", "Recall", "F1"],
    var_name="Metric", value_name="Score",
)

g = sns.catplot(
    data=bert_melted, kind="bar",
    x="Ablation", y="Score", hue="Model", col="Metric",
    palette="viridis", height=4, aspect=0.9,
)
g.fig.suptitle("BERTScore by (Model x Ablation)", y=1.02)
plt.tight_layout()
plt.savefig("data/eval_results/bert_scores_by_ablation.png", bbox_inches="tight")
plt.show()

In [ ]:
rouge_melted = system_rouge_df.melt(
    id_vars=["Model", "Ablation"],
    value_vars=["ROUGE-1", "ROUGE-2", "ROUGE-L", "ROUGE-Lsum"],
    var_name="Metric", value_name="Score",
)

g = sns.catplot(
    data=rouge_melted, kind="bar",
    x="Ablation", y="Score", hue="Model", col="Metric",
    palette="magma", height=4, aspect=0.9,
)
g.fig.suptitle("ROUGE by (Model x Ablation)", y=1.02)
plt.tight_layout()
plt.savefig("data/eval_results/rouge_scores_by_ablation.png", bbox_inches="tight")
plt.show()

### Error Analysis: Identifying Low-Scoring Summaries

Examine some of the summaries that received the lowest BERT F1 scores to understand their failure modes. This manual inspection is crucial for qualitative error analysis.

In [ ]:
# Worst 10 samples by BERT F1, tagged with (model, ablation) for diagnosis
rows = []
for (model, ab), df in results.items():
    f1s = per_sample_bert[(model, ab)]["f1"]
    for i, f1 in enumerate(f1s):
        rows.append({
            "Model": model,
            "Ablation": ab,
            "row_idx": i,
            "bert_f1": f1,
            "original_text":     df.iloc[i].get("original_text", ""),
            "reference_summary": df.iloc[i].get("reference_summary", ""),
            "generated_summary": df.iloc[i].get("generated_summary", ""),
        })

all_samples_df = pd.DataFrame(rows)
lowest_f1 = all_samples_df.sort_values("bert_f1", ascending=True).head(10)

print("Top 10 lowest-BERT-F1 samples across all combinations:")
display(lowest_f1[["Model", "Ablation", "bert_f1", "generated_summary", "reference_summary"]])

all_samples_df.to_csv("data/eval_results/per_sample_bert.csv", index=False)
lowest_f1.to_csv("data/eval_results/error_analysis_lowest_f1.csv", index=False)
print("Saved: data/eval_results/per_sample_bert.csv")
print("Saved: data/eval_results/error_analysis_lowest_f1.csv")